# 05 · Execution evaluation

**What this notebook decides.** Whether fine-tuning improved executable SQL
generation, and by how much, with an honest interval around the answer.

This is the notebook the whole repository exists to produce. It reports a
difference, not two accuracies side by side, because the comparison is paired:
both models were scored on the same examples, and pairing removes the
example-difficulty variance that otherwise swamps the effect.

In [ ]:
import pandas as pd

from qwen_text2sql.reporting import (
    artifact,
    compare_models,
    environment_report,
    headline_metrics,
    load_records,
    preflight,
    project_root,
)
from qwen_text2sql.reporting.style import use_house_style

ROOT = project_root()
PALETTE = use_house_style("light")
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 140)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")
ROOT

In [ ]:
report = environment_report()
print(f"commit   {report['git_commit']}")
print(f"python   {report['python']}")
print(f"platform {report['platform']}")
pd.Series(report["packages"], name="version").to_frame()

In [ ]:
pd.DataFrame(preflight("baseline_records", "lora_records"))

## 1 · Each run on its own

In [ ]:
RUNS = {
    "baseline (A)": "baseline_records",
    "lora (B)": "lora_records",
}

runs = {name: load_records(artifact(key).require()) for name, key in RUNS.items()}

per_run = pd.concat(
    {name: headline_metrics(frame).set_index("metric") for name, frame in runs.items()},
    names=["run"],
)
per_run.style.format({"estimate": "{:.1%}", "lower": "{:.1%}", "upper": "{:.1%}"}, na_rep="—")

Overlapping intervals here are **not** evidence of "no difference". Two
overlapping marginal intervals routinely accompany a clearly significant paired
difference, because the pairing information is thrown away when each run is
summarised alone. The next cell is the one that answers the question.

## 2 · The paired comparison

In [ ]:
comparison = compare_models(runs, baseline="baseline (A)", n_bootstrap=10_000, seed=42)
comparison.style.format(
    {
        "accuracy": "{:.1%}",
        "baseline_accuracy": "{:.1%}",
        "difference": "{:+.1%}",
        "lower": "{:+.1%}",
        "upper": "{:+.1%}",
        "probability_positive": "{:.3f}",
    }
)

In [ ]:
from qwen_text2sql.reporting import figures

figures.model_comparison(comparison)

**How to read the interval.** It is a 95% percentile bootstrap interval for the
paired difference. If it contains zero, this evaluation does not distinguish the
two models — which is a finding, not a failure, and is exactly what the power
analysis in [`00`](00_research_protocol.ipynb) was for.

**`fixed` and `broke` matter as much as the net.** A +3 point net difference
built from 40 fixes and 25 regressions is a model that changed behaviour
substantially; one built from 4 fixes and 1 regression is a model that barely
moved. The two justify very different next experiments.

In [ ]:
churn = comparison[["model", "n", "fixed", "broke", "difference"]].copy()
churn["churn_rate"] = (churn["fixed"] + churn["broke"]) / churn["n"]
churn["net_of_churn"] = (churn["fixed"] - churn["broke"]) / (churn["fixed"] + churn["broke"])
churn.style.format({"difference": "{:+.1%}", "churn_rate": "{:.1%}", "net_of_churn": "{:+.2f}"})

`churn_rate` is the discordance rate the power analysis assumed. If it is far
below the assumed 30%, the comparison has less information than planned and the
interval will be correspondingly wide.

## 3 · Where the gain comes from

An aggregate improvement can be uniform across schemas, or concentrated in a few
databases. Only the first generalises.

In [ ]:
from qwen_text2sql.reporting import accuracy_by_database

per_db = {name: accuracy_by_database(frame) for name, frame in runs.items()}
figures.accuracy_by_database(per_db["lora (B)"], top=25)

In [ ]:
merged = (
    per_db["lora (B)"][["db_id", "n", "execution_accuracy"]]
    .merge(
        per_db["baseline (A)"][["db_id", "execution_accuracy"]],
        on="db_id",
        suffixes=("_lora", "_baseline"),
    )
    .assign(
        delta=lambda frame: frame["execution_accuracy_lora"] - frame["execution_accuracy_baseline"]
    )
    .sort_values("delta", ascending=False)
)
summary = pd.Series(
    {
        "databases improved": int((merged["delta"] > 0).sum()),
        "databases unchanged": int((merged["delta"] == 0).sum()),
        "databases regressed": int((merged["delta"] < 0).sum()),
        "median per-database delta": merged["delta"].median(),
    },
    name="value",
).to_frame()
display(summary)
display(merged.head(10))

## 4 · Latency

Adapters are supposed to be cheap. This checks it rather than assuming it.

In [ ]:
latency = pd.DataFrame(
    {
        name: frame["latency_ms"].describe(percentiles=[0.5, 0.9, 0.99])
        for name, frame in runs.items()
    }
)
latency

## Reading this result

**What it shows.** A paired difference in execution accuracy with a bootstrap
interval, how that difference decomposes into fixes and regressions, and whether
it is spread across databases or concentrated in a few.

**What it does not show.**

- **Generalisation beyond BIRD.** These are BIRD databases with BIRD-style
  questions. Treat the revised development set as the external check.
- **Semantic correctness.** Execution equivalence is a proxy. Two queries can
  agree on this database and disagree on another, and the comparator treats rows
  as a multiset with a numerical tolerance — deliberately transparent, but not
  the official evaluator. For a leaderboard claim, run the benchmark's official
  evaluator as an additional external check.
- **A ranking of methods in general.** One seed, one prompt, one evaluation set.

---

### Provenance

Every number above is produced by code in `src/qwen_text2sql/`, which is unit
tested and type checked. This notebook orchestrates and interprets; it does not
define the analysis. If a figure here looks wrong, the fix belongs in `src/` and
comes with a test.